# Two Traders with Private Signals

Two traders trade one asset. Each observes a private signal, its own inventory and the common price, and the price
drift depends on both signals and both trading rates.

- Signals: $d\alpha^i_t = -\kappa\,\alpha^i_t\,dt + \sigma_\alpha\,dW^i_t$, observed by trader $i$ only.
- Price: $dS_t = \mu_t\,dt + \sigma\,dW^S_t$, with $\mu_t = \alpha^1_t + \alpha^2_t + \lambda(\nu^1_t + \nu^2_t)$.
- Inventories: $dq^i_t = \nu^i_t\,dt$.
- Criterion: trader $i$ maximizes $\mathbb{E}\big[\int_0^T (q^i_t \mu_t - a(\nu^i_t)^2 - \phi(q^i_t)^2)\,dt - \psi(q^i_T)^2\big]$ over $\nu^i$ adapted to the filtration of $(\alpha^i, q^i, S)$.

Trader $i$ infers the other signal from the price, which also moves with the other trader's trades, and those trades
depend on the other trader's inference of $\alpha^i$. No finite-dimensional filter closes this hierarchy. The game is
solved on the Euler--Maruyama grid with $N = 50$ steps on $[0, 1]$.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import TwoTraders, LinearStrategies
game = TwoTraders(lam=1.0)

## 1. Exact solution in the linear class

In this Gaussian linear-quadratic game, a best response to a strategy linear in the observed path is itself linear.
The equilibrium therefore lies in the class
$\nu^i_{t_j} = \sum_{k \le j} \big(\beta^i_{jk,0}\,\alpha^i_{t_k} + \beta^i_{jk,1}\,S_{t_k}\big)$.
Every process is then linear in the Gaussian shocks of the discretised game, so the expected rewards are computed
exactly, without simulation. `LinearStrategies` iterates damped best responses over the coefficients, each
holding the other trader's control fixed as a process, as in the open-loop equilibrium of the note.

In [ ]:
exact_solver = LinearStrategies(game)
beta = exact_solver.solve()
validation = validation_paths(game)                                # the validation paths of every Solver
exact = to_players(exact_solver.controls(beta, validation))
print(f'mean |nu| of the exact equilibrium: {np.mean([float(e.abs().mean()) for e in exact]):.4f}')

## 2. The learned open-loop equilibrium

The learned method computes the open-loop equilibrium by damped Picard iteration. Each trader's control is a network
of its own observations.

- Control: the network $\mathcal{N}^i$ of the note. A GRU, a recurrent network, takes the rescaled increments of
  $(\alpha^i, S)$ as input and keeps a learned summary $h_j \in \mathbb{R}^{32}$ of the observed path up to $t_j$. A
  width-64 feedforward network maps $(t_j, \alpha^i_{t_j}, q^i_{t_j}, S_{t_j}, h_j)$ to $\nu^i_{t_j}$.
- Target: $\Lambda^i / (2a) = \nu^i + G^i / (2a\,\Delta t)$, with $G^i$ the derivative of the realised reward with respect
  to $\nu^i$, computed by automatic differentiation with the other trader's controls held fixed.
- Regression: minimizing a mean squared error between the control and the damped target, with damping 0.6 and a
  martingale control variate subtracted from the target.
- Budget: 16384 paths, 750 Adam steps per iteration, 10 iterations, controls averaged from iteration 4.

In [ ]:
solver = Solver(game, damping=0.6, steps=750, batch=2048, n_paths=16384, control_variate=True)
learned, history = solver.run(10, average_from=4, reference=exact)
print(f'relative L2 error against the exact equilibrium: {relative_l2(learned[0], exact[0]):.3f}, {relative_l2(learned[1], exact[1]):.3f}')

In [ ]:
plot_controls(learned, exact, game.dt, ['trader 1', 'trader 2'], ylabel=r'$\nu_t$')